# Day 10：模块、包与异常处理（Jupyter Notebook 实战手册）

> **所属阶段**：第 1 阶段 · 第一阶段：Python 编程基础 ｜ **第 2 周**：第 2 周：面向对象、工程化与命令行 AI 助手实战  
> **GitLab 工单**：`STARMIND-010` ｜ **开发分支**：`feat/day10-modules-exceptions-venv`  
> **配套 HTML5 课件**：`courseware/day10/index.html` ｜ **独立版本快照**：`versions/day10/`

---

## 🎙️ 老陈开场旁白

聊个真事：去年隔壁部门一个新人把 `try: ... except: pass` 写进了线上 AI 服务里，结果 API 密钥过期了整整两天，用户全收到空回复，监控一条报警都没发，复盘会被批得体无完肤！记住老陈的红线：**裸 `except:` 吞掉所有异常是大厂头号大忌！**
今天 STARMIND-010 咱们干两件让项目真正具备「工程化骨架」的事：
1）吃透 Python 的模块（Module）、包（Package / `__init__.py`）、虚拟环境 `venv` 与 `requirements.txt` 依赖锁规范；
2）为星辰智脑设计一套层次清晰的自定义异常树（`StarMindError -> RateLimitError / AuthError / ContextLengthError`），并写出支持「主模型限流自动切备用模型」的容错调度器！

### 承上启下
- **昨日基线**：昨天咱们用继承和多态搭出了多模型适配层，但目前还没处理调用超时、限流、密钥无效等生产异常。
- **明日预告**：明天（Day 11）咱们学习文件操作（txt/csv/json）与 os/pathlib/datetime/re 常用标准库，给星辰智脑打造批量文档读取与关键词统计工具。


In [ ]:
# 环境自检与路径初始化：确保可以无缝导入当日版本模块并支持异步协程运行
import os
import sys
from pathlib import Path

# 允许在 Jupyter 已有事件循环内安全调用 asyncio.run()
try:
    import nest_asyncio
    nest_asyncio.apply()
except ImportError:
    pass

PROJECT_ROOT = Path.cwd()
while PROJECT_ROOT != PROJECT_ROOT.parent and not (PROJECT_ROOT / 'versions').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

DAY_DIR = PROJECT_ROOT / 'versions' / 'day10'
for _p in (DAY_DIR, DAY_DIR / 'src', PROJECT_ROOT):
    if _p.exists() and str(_p) not in sys.path:
        sys.path.insert(0, str(_p))
os.chdir(DAY_DIR)
__file__ = str(DAY_DIR / 'src' / 'notebook_entry.py')

print(f'✅ Python 版本: {sys.version.split()[0]}')
print(f'✅ 项目根目录: {PROJECT_ROOT}')
print(f'✅ 当日版本目录: {DAY_DIR}')


## 📋 企业需求背景与验收目标（STARMIND-010）

星辰智脑接入多模型后，生产环境频繁面临上游 429 限流、401 密钥失效、输入超长等故障，需要统一的自定义异常体系、优雅降级机制与标准化依赖隔离方案。

**今日迭代目标：**
- 掌握 `import` 搜索路径、包结构（`__init__.py`）、`venv` 虚拟环境与 `requirements.txt` 管理。
- 熟练运用 `try / except / else / finally` 与 `raise ... from exc` 异常链保留原始堆栈。
- 设计 `StarMindError` 自定义异常层级并实现主备模型自动降级执行器。

**验收标准：**
- 自定义异常树包含 `StarMindError` 基类及 `AuthError`、`RateLimitError`、`ModelTimeoutError` 子类，携带 `error_code` 与 `retryable` 标记。
- 容错执行器在主模型抛出 `RateLimitError` 时自动无缝切换至备用模型；遇到 `AuthError` 时立即向上抛出不盲目降级。
- 依赖清单解析工具能校验 `requirements.txt` 是否锁定版本号，pytest 全绿。

---

## 📚 课堂核心技术笔记

### 1. 1. 模块、包与 __name__ == '__main__'

一个 `.py` 文件就是一个模块（Module）；包含 `__init__.py` 的文件夹就是一个包（Package）。每个模块都应把演示代码放进 `if __name__ == '__main__':` 块里，否则别人 `import` 你的模块时会意外执行你的演示打印甚至触发网络请求！

> **⚠️ 老陈避坑**：切忌把自己的业务文件命名为 `json.py`、`random.py` 或 `requests.py`！那会遮蔽（Shadow）同名标准库或第三方库，引发极其诡异的 `AttributeError`。

### 2. 2. try / except / else / finally 完整语义

- `try`：放可能出错的最小范围代码；
- `except SpecificError as e`：精准捕获并处理；
- `else`：仅当 `try` 块**完全没有异常**时执行；
- `finally`：**无论是否发生异常（甚至有 return）**都必然执行，用来关闭文件、释放锁、记录审计日志。

> **⚠️ 老陈避坑**：重新包装异常时务必使用 `raise StarMindError('调用失败') from orig_exc`，这样 traceback 会保留原始报错链，排查效率翻倍。

### 3. 3. venv 虚拟环境与 requirements.txt

标准三步：`python3 -m venv .venv` 创建隔离环境 -> 激活后安装依赖 -> `pip freeze > requirements.txt` 锁定版本。团队新人拉下代码后只需 `pip install -r requirements.txt` 即可 100% 复现一致环境。

> **⚠️ 老陈避坑**：千万别在 `requirements.txt` 里只写包名不锁版本（如只写 `pydantic`），某天上游发了个不兼容大版本，你的线上镜像一重新构建直接挂掉！


## 💻 实操步骤 1：步骤一：构建星辰智脑自定义异常体系

- **目标文件**：`versions/day10/src/day10_exceptions.py`
- **老陈代码旁白**：先建异常树！给每个异常挂上标准错误码 `code` 和 `retryable` 标记，让下游调用方一眼看懂该不该重试或降级。


In [ ]:
"""
STARMIND-010 模块一：星辰智脑自定义异常体系。
演示异常类继承、结构化错误元数据（code/retryable）以及标准错误字典序列化。
"""

from typing import Dict, Any


class StarMindError(Exception):
    """星辰智脑平台所有业务异常的统一基类。"""

    code: str = "E500"
    retryable: bool = False

    def __init__(self, message: str, detail: str = "") -> None:
        super().__init__(message)
        self.message = message
        self.detail = detail

    def to_error_dict(self) -> Dict[str, Any]:
        """将异常转为前后端统一的 JSON 错误响应结构。"""
        return {
            "ok": False,
            "error_code": self.code,
            "message": self.message,
            "detail": self.detail,
            "retryable": self.retryable,
        }


class RateLimitError(StarMindError):
    """上游大模型触发限流（HTTP 429），支持退避重试或切备用模型。"""

    code = "E429"
    retryable = True


class ModelTimeoutError(StarMindError):
    """上游大模型推理超时（HTTP 504），支持降级重试。"""

    code = "E504"
    retryable = True


class AuthError(StarMindError):
    """API Key 无效或已过期（HTTP 401），不可重试，需立即告警人工介入。"""

    code = "E401"
    retryable = False


if __name__ == "__main__":
    err = RateLimitError("DeepSeek 主通道触发 TPM 限流", detail="retry-after: 3s")
    print("异常字典:", err.to_error_dict())


### 🔍 步骤 1 代码深度解析

给异常加上 `to_error_dict()` 后，后续在 Day 23 写 FastAPI 接口时，全局异常处理器可以直接把异常对象一键转成规范的 JSON 错误响应。

**预期运行输出参考：**
```text
异常字典: {'ok': False, 'error_code': 'E429', 'message': 'DeepSeek 主通道触发 TPM 限流', 'detail': 'retry-after: 3s', 'retryable': True}
```


## 💻 实操步骤 2：步骤二：实现主备模型容错降级器与 requirements.txt 审计工具

- **目标文件**：`versions/day10/src/day10_resilient_runner.py`
- **老陈代码旁白**：现在把 `try / except / else / finally` 用到极致：主模型限流或超时自动切备用模型，401 鉴权错立刻熔断报警，同时附带一个 CI 依赖锁检查器。


In [ ]:
"""
STARMIND-010 模块二：主备模型容错执行器与 requirements.txt 规范审计工具。
演示 try/except/else/finally、精准异常分流、异常链 raise...from 与工程依赖检查。
"""

from typing import Callable, Dict, Any, List
from day10_exceptions import StarMindError, AuthError


def call_with_fallback(
    primary_fn: Callable[[str], str],
    fallback_fn: Callable[[str], str],
    prompt: str,
) -> Dict[str, Any]:
    """
    高可用模型调用调度器：
    - 主模型正常：直接返回 primary 结果；
    - 主模型抛出可重试 StarMindError（如 RateLimitError/Timeout）：自动降级至备用模型；
    - 主模型抛出 AuthError 等不可重试异常：立即透传上抛，绝不掩盖根因。
    """
    audit_events: List[str] = []
    try:
        audit_events.append("try_primary")
        reply = primary_fn(prompt)
    except AuthError:
        # 鉴权失败属于配置事故，不降级直接暴露
        audit_events.append("auth_abort")
        raise
    except StarMindError as exc:
        if not exc.retryable:
            audit_events.append("non_retryable_abort")
            raise
        # 可重试/可降级异常：自动切换备用模型
        audit_events.append(f"fallback_on_{exc.code}")
        try:
            reply = fallback_fn(prompt)
            channel = "fallback"
        except Exception as fb_exc:
            raise StarMindError("主备模型均调用失败", detail=str(fb_exc)) from fb_exc
    else:
        # 仅当 try 块完全无异常时执行
        channel = "primary"
        audit_events.append("primary_ok")
    finally:
        # 无论成功、降级还是抛异常，审计收尾钩子必然执行
        audit_events.append("audit_closed")

    return {
        "ok": True,
        "channel": channel,
        "reply": reply,
        "audit_events": audit_events,
    }


def audit_requirements_text(req_content: str) -> Dict[str, List[str]]:
    """检查 requirements.txt 文本，识别已锁版本与未锁版本的风险依赖。"""
    pinned: List[str] = []
    unpinned: List[str] = []
    for raw_line in req_content.splitlines():
        line = raw_line.strip()
        if not line or line.startswith("#"):
            continue
        if "==" in line:
            pinned.append(line)
        else:
            unpinned.append(line)
    return {"pinned": pinned, "unpinned": unpinned}


if __name__ == "__main__":
    from day10_exceptions import RateLimitError

    def mock_primary(_p: str) -> str:
        raise RateLimitError("主通道 429 限流")

    def mock_fallback(p: str) -> str:
        return f"[备用通义千问兜底] 已处理: {p}"

    print("降级结果:", call_with_fallback(mock_primary, mock_fallback, "查询报销流程"))
    sample_req = "# 核心依赖\nfastapi==0.115.0\npydantic\nuvicorn==0.30.0\n"
    print("依赖审计:", audit_requirements_text(sample_req))


### 🔍 步骤 2 代码深度解析

1. 注意 `except AuthError:` 写在 `except StarMindError:` 前面——因为 `AuthError` 是 `StarMindError` 的子类，异常捕获必须遵循「先子类、后父类」的顺序。
2. `raise StarMindError(...) from fb_exc` 保留了备用模型崩溃的原始原因，线上排障一目了然。

**预期运行输出参考：**
```text
降级结果: {'ok': True, 'channel': 'fallback', 'reply': '[备用通义千问兜底] 已处理: 查询报销流程', 'audit_events': ['try_primary', 'fallback_on_E429', 'audit_closed']}
依赖审计: {'pinned': ['fastapi==0.115.0', 'uvicorn==0.30.0'], 'unpinned': ['pydantic']}
```


## 🧪 当日自动化单元测试

在终端运行 `pytest versions/day10/tests/test_day10.py -v` 即可验收全部测试断言。下面展示完整测试脚本源码，供对照学习测试写法：

```python
"""STARMIND-010 自动化单元测试（含故障注入）。"""

import sys
from pathlib import Path
import pytest

ROOT = Path(__file__).resolve().parent.parent
for p in (ROOT, ROOT / "src"):
    if str(p) not in sys.path:
        sys.path.insert(0, str(p))

from day10_exceptions import StarMindError, RateLimitError, ModelTimeoutError, AuthError
from day10_resilient_runner import call_with_fallback, audit_requirements_text


def test_primary_success():
    res = call_with_fallback(lambda p: f"P:{p}", lambda p: f"F:{p}", "hi")
    assert res["channel"] == "primary"
    assert res["reply"] == "P:hi"
    assert res["audit_events"] == ["try_primary", "primary_ok", "audit_closed"]


def test_fallback_on_rate_limit_and_timeout():
    def fail_429(_p):
        raise RateLimitError("too many requests")

    def fail_504(_p):
        raise ModelTimeoutError("gateway timeout")

    r1 = call_with_fallback(fail_429, lambda p: f"BACKUP:{p}", "hello")
    assert r1["channel"] == "fallback"
    assert "fallback_on_E429" in r1["audit_events"]

    r2 = call_with_fallback(fail_504, lambda p: f"BACKUP:{p}", "hello")
    assert r2["channel"] == "fallback"
    assert "fallback_on_E504" in r2["audit_events"]


def test_auth_error_never_falls_back():
    def fail_401(_p):
        raise AuthError("invalid key")

    with pytest.raises(AuthError) as exc_info:
        call_with_fallback(fail_401, lambda p: "should not run", "hello")
    assert exc_info.value.to_error_dict()["error_code"] == "E401"


def test_audit_requirements_text():
    res = audit_requirements_text("# comment\npytest==8.3.0\nhttpx\n")
    assert res["pinned"] == ["pytest==8.3.0"]
    assert res["unpinned"] == ["httpx"]

```


## 📝 课后梯度实战作业与标准答案

建议先在下方新建空白单元格独立完成编码，运行通过后再核对参考答案。


### 作业 1（基础巩固）：基础巩固：安全数值解析器

**题目要求：**
编写 `safe_parse_int(value: str, default: int = 0) -> int`，尝试将字符串转为整数，捕获 `ValueError` 和 `TypeError` 返回默认值（绝不使用裸 `except:`）。

**思路提示：** 使用 `except (ValueError, TypeError):` 同时精准捕获两种异常。


In [ ]:
"""Day 10 课后题 1：精准异常捕获的安全整数解析器。"""


def safe_parse_int(value: object, default: int = 0) -> int:
    """将任意输入安全解析为整数，失败时返回 default。"""
    try:
        return int(value)  # type: ignore[arg-type]
    except (ValueError, TypeError):
        return default


if __name__ == "__main__":
    for sample in ["42", "3.14", "abc", None]:
        print(f"{sample!r:8s} -> {safe_parse_int(sample, default=-1)}")


**答案解析与评分标准：**

演示元组形式同时捕获多种指定异常 `except (ValueError, TypeError)`，既安全又不会误吞 `KeyboardInterrupt`。


### 作业 2（企业实战）：企业实战：带最大重试次数的同步重试函数

**题目要求：**
编写 `retry_call(fn, max_attempts=3)`，当 `fn()` 抛出 `TimeoutError` 时自动重试，若超过最大次数仍失败则将最后一次异常抛出。

**思路提示：** 循环 `range(1, max_attempts + 1)`，在最后一次迭代直接 `raise`。


In [ ]:
"""Day 10 课后题 2：指定异常重试执行器。"""

from typing import Callable, TypeVar

T = TypeVar("T")


def retry_call(fn: Callable[[], T], max_attempts: int = 3) -> T:
    """遇到 TimeoutError 时自动重试，最多执行 max_attempts 次。"""
    for attempt in range(1, max_attempts + 1):
        try:
            return fn()
        except TimeoutError:
            if attempt == max_attempts:
                raise
    raise RuntimeError("unreachable")


if __name__ == "__main__":
    state = {"count": 0}

    def flaky_api() -> str:
        state["count"] += 1
        if state["count"] < 3:
            raise TimeoutError(f"第 {state['count']} 次超时")
        return "第 3 次调用成功！"

    print(retry_call(flaky_api, max_attempts=3))


**答案解析与评分标准：**

这是后续 Day 13 装饰器版重试与 `tenacity` 库的原型，注意最后一次失败必须原样 `raise` 保留完整堆栈。


### 作业 3（架构进阶）：架构进阶：自定义上下文超长异常与自动截断恢复

**题目要求：**
定义 `ContextOverflowError(StarMindError)` 携带 `max_chars` 属性；编写调用包装器，当捕获到该异常时自动将 prompt 截断至 `max_chars` 后重新调用一次。

**思路提示：** 捕获 `ContextOverflowError as e:` 后用 `prompt[:e.max_chars]` 二次尝试。


In [ ]:
"""Day 10 课后题 3：上下文超长自愈恢复机制。"""


class ContextOverflowError(Exception):
    def __init__(self, max_chars: int) -> None:
        super().__init__(f"上下文超过最大长度 {max_chars}")
        self.max_chars = max_chars


def invoke_with_auto_truncate(model_fn, prompt: str) -> str:
    try:
        return model_fn(prompt)
    except ContextOverflowError as exc:
        trimmed = prompt[: exc.max_chars]
        return model_fn(trimmed)


if __name__ == "__main__":
    def strict_model(text: str) -> str:
        if len(text) > 10:
            raise ContextOverflowError(max_chars=10)
        return f"OK({text})"

    print(invoke_with_auto_truncate(strict_model, "这是一段超过十个字符的长提问文本"))


**答案解析与评分标准：**

异常对象携带业务参数 `max_chars`，让外层容错逻辑具备「自愈能力」，非常贴近真实 LLM 网关处理超长上下文的做法。


---

## 🎯 老陈收尾复盘

今天这套自定义异常体系 + 自动降级容错器，才是真正的企业级 AI 工程思维。记住三条铁律：1）异常要精准捕获，绝不 `except: pass`；2）区分可重试/可降级异常（如 429 限流）与致命异常（如 401 鉴权失败）；3）每个独立项目必建专属 `venv` 并导出 `requirements.txt`。
